In [36]:
# packages
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt

Nous allons, avant d'écrire l'algorithme, écrire les fonctions importantes, à savoir: contraction de tenseurs, décomposition QR, SVD, diagonalisation

Un tenseur d'ordre N (ici N=3) sera représenté par un tableau à N dimensions. Ainsi, pour N=3, on aura des tableaux de tableaux de tableaux.

In [30]:
n=2 #nombre de sites
d=2 #indice visible, branches libres du MPS
D=3 #indice de liaison, branches liées du MPS

In [27]:
def into_matrix_left(t): #transformer un tenseur d'ordre 3 en matrice pour appliquer SVD/QR en regroupant les 2 premiers indices
    return np.reshape(t,(D*d,D))

In [28]:
def into_matrix_right(t): #transformer un tenseur d'ordre 3 en matrice pour appliquer SVD/QR en regroupant les 2 derniers indices
    return np.reshape(t,(D,d*D))

In [29]:
def from_matrix(m): #revenir au tenseur d'ordre 3 après SVD/QR
    return np.reshape(m,(D,d,D))

In [37]:
def QR_left(t): #décomposition QR
    tbis=t.copy()
    q,r=sc.linalg.qr(into_matrix_left(tbis),mode='economic')
    return from_matrix(q),r

In [38]:
def QR_right(t): #décomposition QR
    tbis=t.copy()
    q,r=sc.linalg.qr(np.transpose(into_matrix_right(tbis)),mode='economic')
    return np.transpose(r),from_matrix(np.transpose(q))

In [ ]:
def MPS_orth_left(M): #orthonormalise le MPS à gauche, M est la liste des n tenseurs
    q,r=sc.linalg.qr(M[0],mode='economic') #On n'utilise pas QR car le 1er tenseur est d'ordre 2
    M[0],M[1]=q,np.tensordot(r,M[1],1)
    for i in range(1,n-1):
        q,r=QR_left(M[i])
        M[i],M[i+1]=q,np.tensordot(r,M[i+1],1)
    return M #On ne s'occupe pas du dernier tenseur car on commencera par le modifier dans l'algorithme

In [47]:
def MPS_orth_right(M): #orthonormalise le MPS à droite, M est la liste des n tenseurs
    q,r=sc.linalg.qr(np.transpose(M[n-1]),mode='economic') #On n'utilise pas QR car le dernier tenseur est d'ordre 2
    M[n-1],M[n-2]=np.transpose(q),np.tensordot(M[n-2],np.transpose(r),1)
    for i in range(n-2,0,-1):
        r,q=QR_right(M[i])
        M[i],M[i-1]=q,np.tensordot(M[i-1],r,1)
    return M #On ne s'occupe pas du premier tenseur car on commencera par le modifier dans l'algorithme

In [55]:
def SVD_left(t): #SVD avec tenseur orthonormal à gauche, sera utilisé pour parcourir le MPS de gauche à droite
    t=into_matrix_left(t)
    U,s,V=sc.linalg.svd(t,full_matrices=False)
    S=np.diag(s)
    return from_matrix(U), np.tensordot(S,V)

In [ ]:
def SVD_right(t): #SVD avec tenseur orthonormal à droite, sera utilisé pour parcourir le MPS de droite à gauche
    t=into_matrix_right(t)
    U,s,V=sc.linalg.svd(t,full_matrices=False)
    S=np.diag(s)
    return np.tensordot(U,S),from_matrix(V)

In [39]:
M=np.array([ [[5, 1, 0],
[3, 3, 2]],
[[1, 1, 0],
[5, 1, 7]],
[[0, 0, 1],
[8, 1, 9]] ])

In [50]:
r=np.array([[0,1,2],[3,4,5],[6,7,8]])

In [52]:
sc.linalg.svd(r,full_matrices=False)

(array([[-0.13511895,  0.90281571,  0.40824829],
        [-0.49633514,  0.29493179, -0.81649658],
        [-0.85755134, -0.31295213,  0.40824829]]),
 array([1.42267074e+01, 1.26522599e+00, 5.89938022e-16]),
 array([[-0.4663281 , -0.57099079, -0.67565348],
        [-0.78477477, -0.08545673,  0.61386131],
        [-0.40824829,  0.81649658, -0.40824829]]))

In [41]:
QR_right(M)

(array([[-6.92820323,  0.        ,  0.        ],
        [-5.48482756, -6.8495742 ,  0.        ],
        [-6.49519053, -9.98237234, -2.27260697]]),
 array([[[-0.72168784, -0.14433757, -0.        ],
         [-0.4330127 , -0.4330127 , -0.28867513]],
 
        [[ 0.43190033, -0.03041552,  0.        ],
         [-0.3832355 ,  0.20074241, -0.79080342]],
 
        [[ 0.16549721,  0.54612127, -0.44002329],
         [-0.59926795, -0.08421405,  0.33841933]]]))